In [17]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks

In [18]:
class Dataset_converter(Dataset):
    def __init__(self, data, token_size, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), token_size), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, token_size*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), token_size))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*token_size:(kk+1)*token_size] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [53]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers=2, num_layers_sleep=2, output_size=7):
        super(brain, self).__init__()

        self.sleep_output_size = sleep_output_size
        self.rnn = nn.RNN(input_size+sleep_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_rnn = nn.RNN(input_size, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, sleep_output_size)
        self.wake_fc = nn.Linear(hidden_wake_size, output_size)

    def forward(self, x, x_=None, hw=None, hs=None, sleep=False, short_term_memory=1, working_memory=1):
        # print(x.shape, 'x')
        if sleep:
            if hs == None:
                out, hs = self.sleep_rnn(x_)
            else:
                out, hs = self.sleep_rnn(x_, hs)
            # print(out.shape)
            sleep_out = self.sleep_fc(out)
        else:
            sleep_out = torch.zeros((1,short_term_memory,self.sleep_output_size))
            
        x = torch.cat((x,sleep_out), dim=2)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        out = self.wake_fc(out[:,-1,:])

        if sleep:
            return out, hw, hs
        else:
            return out, hw


In [58]:
def wake_period(model, train_data, test_data, token_size, working_memory, short_term_memory, lr=4e-4):
    data_set = Dataset_converter(train_data, token_size, working_memory, short_term_memory)
    train_loader = DataLoader(data_set, batch_size=1, shuffle=False)
    data_set = Dataset_converter(test_data, token_size, working_memory, short_term_memory)
    test_loader = DataLoader(data_set, batch_size=1, shuffle=False)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95)
    criterion = torch.nn.CrossEntropyLoss()

    total = 0
    correct_train = np.zeros(1000,dtype=float)
    correct_test = np.zeros(1000,dtype=float)
    train_acc = []
    test_acc = []
    for (X_train, y_train), (X_test, y_test) in zip(train_loader, test_loader):
        optimizer.zero_grad()
    
        if total == 0:
            predicted_y, hidden = model(X_train, short_term_memory=short_term_memory)
        else:
            predicted_y, hidden = model(X_train, hw=mem, short_term_memory=short_term_memory)

        #print(predicted_y.shape, y_train.shape)
        loss = criterion(predicted_y, y_train)
        loss.backward(retain_graph=True)
        optimizer.step()

        with torch.no_grad():
            mem=hidden.clone()
            true_y = y_train.argmax(axis=1)
            estimated_y = predicted_y.argmax(axis=1)

            if total == 0:
                predicted_y_test, hidden_ = model(X_test, short_term_memory=short_term_memory)
            else:
                predicted_y_test, hidden_ = model(X_test, hw=hidden_, short_term_memory=short_term_memory)

            true_y_test = y_test.argmax(axis=1)
            estimated_y_test = predicted_y_test.argmax(axis=1)

            total += 1
            if true_y == estimated_y:
                correct_train[total%1000] = 1
            else:
                correct_train[total%1000] = 0


            if true_y_test == estimated_y_test:
                correct_test[total%1000] = 1
            else:
                correct_test[total%1000] = 0


            train_acc.append(
                np.sum(correct_train)/total if total<1000 else np.sum(correct_train)/1000
            )
            test_acc.append(
                np.sum(correct_test)/total if total<1000 else np.sum(correct_test)/1000
            )

            if total%1000 == 0:
                print(f'Iter : {total+1}, loss: {loss:.4f}, train accuracy: {train_acc[-1]:.4f}, test accuracy: {test_acc[-1]:.4f}')

    return model, train_acc, test_acc

In [66]:
### initial training ###
total_samples = 400000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 130
hidden_compressor_size = 10
hidden_sleep_size = 100
sleep_output_size = 10
num_layers_wake = 1
num_layers_sleep = 2
#output_sleep = len(tokens)
#input_size = len(tokens)*working_memory
lr = 4e-4

n_community = 2
n_members = 3

token_size = n_community*n_members+1
input_size = token_size*working_memory

train_data = get_sequence(total_samples, n_community, n_members)
test_data = get_sequence(total_samples, n_community, n_members, train=False)
model = brain(input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers_wake, num_layers_sleep, output_size=token_size)
model, train_acc, test_acc = wake_period(model, train_data, test_data, token_size, working_memory, short_term_memory, lr=lr)

Iter : 1001, loss: 2.0045, train accuracy: 0.2420, test accuracy: 0.2370
Iter : 2001, loss: 1.6803, train accuracy: 0.3090, test accuracy: 0.3040
Iter : 3001, loss: 1.4757, train accuracy: 0.5770, test accuracy: 0.4820
Iter : 4001, loss: 2.1182, train accuracy: 0.6440, test accuracy: 0.5540
Iter : 5001, loss: 1.2346, train accuracy: 0.6590, test accuracy: 0.5540
Iter : 6001, loss: 1.7791, train accuracy: 0.6790, test accuracy: 0.5730
Iter : 7001, loss: 1.5483, train accuracy: 0.6620, test accuracy: 0.5490
Iter : 8001, loss: 1.9194, train accuracy: 0.6830, test accuracy: 0.5600
Iter : 9001, loss: 0.9033, train accuracy: 0.6640, test accuracy: 0.5700
Iter : 10001, loss: 2.3401, train accuracy: 0.6930, test accuracy: 0.5700
Iter : 11001, loss: 0.8749, train accuracy: 0.6910, test accuracy: 0.5870
Iter : 12001, loss: 1.7903, train accuracy: 0.6900, test accuracy: 0.5610
Iter : 13001, loss: 1.5097, train accuracy: 0.6840, test accuracy: 0.5480
Iter : 14001, loss: 1.2735, train accuracy: 0.7

In [38]:
train_data

'...'